# Data Extraction using `spaCy'

Feedback/questions: pushpak.karnick@digipen.edu

### Module Objectives

- Extract tokens from a semi-structured text
- Extract important words from the text
- Extract named-entities and actions
- Establish relationships between entities based on connecting action word(s)

## Extracting tokens
We wish to extract more information from our data than the text blobs under the `<Item>` tags. In this exercise, we shall use the cleaned version of the play _Much Ado About Nothing_ by William Shakespeare as our input dataset.

A quick glance at the data reveals that this file also contains information about the locations and the characters. As a first pass, let us collect all the locations and the characters (and their attributes) into two lists.

In [1]:
import gc
import xml.etree.ElementTree as ET
import re
from collections import Counter
from jupyter_lsp.specs import json

import sys
from pathlib import Path

import spacy
nlp = spacy.load("en_core_web_sm")

cwd = Path.cwd()

src_dir = cwd
while src_dir.name != "src":
    src_dir = src_dir.parent

repo_root = src_dir.parent
sys.path.insert(0, str(src_dir))

train_dir = repo_root / "data" / "train"
file_path = train_dir / "Shakespeare_Much_Ado_About_Nothing.txt"

tree = ET.parse(file_path)
root = tree.getroot()

print("exists?", file_path.exists())
print(file_path)

tree = ET.parse(file_path)
root = tree.getroot()
characters = []

def collectCharacterInfo(root):
    result = []

    cast = root.findall('.//Character')

    for characterInfo in cast:
        full_name = characterInfo.get("name")

        if full_name is None:
            continue

        # Split only on first comma
        parts = full_name.split(",", 1)

        name = parts[0].strip()
        description = parts[1].strip() if len(parts) > 1 else ""

        # Remove trailing period from description
        description = description.rstrip(".")

        result.append({
            "name": name,
            "desc": description
        })

    return result

characters = collectCharacterInfo(root)
print(characters)

exists? True
E:\DigiPenMasterCourse\Semester4_2026\cs592_NLP\cs592-natural-language-processing\Chankasemporn_Ju-ve_CS592_NLP_Project\data\train\Shakespeare_Much_Ado_About_Nothing.txt
[{'name': 'DON PEDRO', 'desc': 'Prince of Arragon'}, {'name': 'DON JOHN', 'desc': 'his bastard Brother'}, {'name': 'CLAUDIO', 'desc': 'a young Lord of Florence'}, {'name': 'BENEDICK', 'desc': 'a young Lord of Padua'}, {'name': 'LEONATO', 'desc': 'Governor of Messina'}, {'name': 'ANTONIO', 'desc': 'his Brother'}, {'name': 'BALTHASAR', 'desc': 'Servant to Don Pedro'}, {'name': 'BORACHIO', 'desc': 'follower of Don John'}, {'name': 'CONRADE', 'desc': 'follower of Don John'}, {'name': 'DOGBERRY', 'desc': 'a Constable'}, {'name': 'VERGES', 'desc': 'a Headborough'}, {'name': 'FRIAR FRANCIS.', 'desc': ''}, {'name': 'A Sexton.', 'desc': ''}, {'name': 'A Boy.', 'desc': ''}, {'name': 'HERO', 'desc': 'Daughter to Leonato'}, {'name': 'BEATRICE', 'desc': 'Niece to Leonato'}, {'name': 'MARGARET', 'desc': 'Waiting gentlew

Similarly, let us collect all the locations

In [2]:
def collectLocationInfo(node):
    """ Collects all locations from the XML tree.

    Args:
        node (xml.etree.ElementTree.Element): Parent node in the XML tree.
            (In your current call you pass root.find('.//Body'), but locations
             are under <Contents>. So if node is None, we safely fall back to root.)

    Returns:
        List[dict]: [{"location": str, "count": int}, ...] sorted by count desc.
    """
    search_root = node if node is not None else root

    raw_locations = []

    # Find every Scene that has a location attribute
    for scene in search_root.findall(".//Scene"):
        loc = scene.get("location")
        if not loc:
            continue

        # Normalize whitespace and remove trailing period
        loc = re.sub(r"\s+", " ", loc).strip().rstrip(".")

        if loc:
            raw_locations.append(loc)

    counts = Counter(raw_locations)

    resultList = [
        {"location": loc, "count": cnt}
        for loc, cnt in counts.most_common()
    ]

    return resultList


locations = collectLocationInfo(root.find(".//Body"))
print(locations)

[{'location': 'Leonato’s Garden', 'count': 3}, {'location': 'A Room in Leonato’s House', 'count': 3}, {'location': 'Before Leonato’s House', 'count': 2}, {'location': 'Another room in Leonato’s house', 'count': 2}, {'location': 'The Inside of a Church', 'count': 2}, {'location': 'A room in Leonato’s house', 'count': 1}, {'location': 'A hall in Leonato’s house', 'count': 1}, {'location': 'A Street', 'count': 1}, {'location': 'Another Room in Leonato’s House', 'count': 1}, {'location': 'A Prison', 'count': 1}]


## Extracting NER information from the character list

Our next step is to feed the information from the `characters` list to `spaCy` and check if it recognizes any of the characters as persons. However, our raw information is in the form of a dictionary, and `spaCy`'s NLP pipeline accepts strings. Thus, we will have to convert the information in the dictionary to a string.

We also know that the language model employed by `spaCy` is English, which means that the input text should be grammatically correct English as much as possible. We employ the following approach to provide `spaCy`'s parser with grammatically correct English:
- We create a string of the form _"Character\_Name is Character\_Description"_.
- We concatenate all such strings into a single text blob.
- This text blob is then fed to `spaCy`'s parser.

In [3]:
import spacy
nlp = spacy.load('en_core_web_sm')

inputText = ""

# Change input data in a way that spaCy can recognize it as valid English sentences.
# Your code here ...
sentences = []

for character in characters:
    name = character.get("name", "").strip()
    desc = character.get("desc", "").strip()

    if not name:
        continue

    if desc:
        sentence = f"{name} is {desc}."
    else:
        sentence = f"{name} is a character."

    sentences.append(sentence)

# Combine into one text blob
inputText = " ".join(sentences)

doc = nlp(inputText.strip())

Let us check the information parsed into the `doc` object.

In [4]:
for token in doc:
    print(
        f"Text: {token.text}, Lemma: {token.lemma_}, POS: {token.pos_}, Tag: {token.tag_}, Dep: {token.dep_}, Shape: {token.shape_}, Is Alpha: {token.is_alpha}, Is Stop: {token.is_stop}")


Text: DON, Lemma: DON, POS: PROPN, Tag: NNP, Dep: compound, Shape: XXX, Is Alpha: True, Is Stop: False
Text: PEDRO, Lemma: PEDRO, POS: PROPN, Tag: NNP, Dep: nsubj, Shape: XXXX, Is Alpha: True, Is Stop: False
Text: is, Lemma: be, POS: AUX, Tag: VBZ, Dep: ROOT, Shape: xx, Is Alpha: True, Is Stop: True
Text: Prince, Lemma: Prince, POS: PROPN, Tag: NNP, Dep: attr, Shape: Xxxxx, Is Alpha: True, Is Stop: False
Text: of, Lemma: of, POS: ADP, Tag: IN, Dep: prep, Shape: xx, Is Alpha: True, Is Stop: True
Text: Arragon, Lemma: Arragon, POS: PROPN, Tag: NNP, Dep: pobj, Shape: Xxxxx, Is Alpha: True, Is Stop: False
Text: ., Lemma: ., POS: PUNCT, Tag: ., Dep: punct, Shape: ., Is Alpha: False, Is Stop: False
Text: DON, Lemma: DON, POS: PROPN, Tag: NNP, Dep: compound, Shape: XXX, Is Alpha: True, Is Stop: False
Text: JOHN, Lemma: JOHN, POS: PROPN, Tag: NNP, Dep: nsubj, Shape: XXXX, Is Alpha: True, Is Stop: False
Text: is, Lemma: be, POS: AUX, Tag: VBZ, Dep: ROOT, Shape: xx, Is Alpha: True, Is Stop: True

In [5]:
from spacy import displacy

# Get the first sentence from doc
first_sentence = list(doc.sents)[0:2]

# Display dependency graph
displacy.render(first_sentence, style='dep', jupyter=True)


So far so good - our text is accepted as valid English sentences and has appropriate dependencies among tokens tracked automatically by `spaCy`. This means our sentences are syntactically valid!

However, a language is more than just its syntax, especially a natural language. Let us check if our input data carries the semantic information we are looking for. In other words, is `spaCy` able to recognize real-world things (or _Named Entities_) from our input?

In [6]:
displacy.render(first_sentence, style='ent', jupyter=True)

### Handling custom NER tokens

From the example above, we can see that neither _Don Pedro_ nor _Don John_ were recognized as a compound token representing a 'PERSON.' We can add custom NER tokens to `spaCy`'s vocabulary to handle such cases.

We have to repeat this process for all named-entities that the default `spaCy` language model is unable to recognize. Some token groups are names of persons, whereas some might be places. We should provide the appropriate NER tag in each case.

This process of adding custom metadata into the existing language model based on our specific data is known as _fine-tuning_ the language model.

Here is the full list of the NER tags supported by `spaCy`:
- PERSON: People, including fictional.
- NORP: Nationalities or religious or political groups.
- FAC: Buildings, airports, highways, bridges, etc.
- ORG: Companies, agencies, institutions, etc.
- GPE: Countries, cities, states.
- LOC: Non-GPE locations, mountain ranges, bodies of water.
- PRODUCT: Objects, vehicles, foods, etc. (Not services.)
- EVENT: Named hurricanes, battles, wars, sports events, etc.
- WORK_OF_ART: Titles of books, songs, etc.
- LAW: Named documents made into laws.
- LANGUAGE: Any named language.
- DATE: Absolute or relative dates or periods.
- TIME: Times smaller than a day.
- PERCENT: Percentage, including "%".
- MONEY: Monetary values, including unit.
- QUANTITY: Measurements, as of weight or distance.
- ORDINAL: "first", "second", etc.
- CARDINAL: Numerals that do not fall under another type.


In [7]:
from spacy.tokens import Span

def merge_tokens_to_ner(doc, token_texts, entity_type="PERSON"):
    """
    Searches for specified tokens in doc and merges them into one NER object.
    
    Args:
        doc: spaCy Doc object
        token_texts: List of token texts to search for and merge (e.g., ["DON", "PEDRO"])
        entity_type: The entity type to assign (default: "PERSON")

    Returns:
        Modified doc object with merged tokens
    """
    normalized_tokens = [t.lower() for t in token_texts]

    # If a single token is provided, find it and add/modify the NER label
    if len(normalized_tokens) == 1:
        for token in doc:
            if token.text.lower() == normalized_tokens[0]:

                new_span = Span(doc, token.i, token.i + 1, label=entity_type)

                # Remove overlapping entities
                new_ents = [
                    ent for ent in doc.ents
                    if not (ent.start <= token.i < ent.end)
                ]

                new_ents.append(new_span)
                doc.set_ents(new_ents)
                return doc

        return doc

    # Check if the token group already exists as a merged entity
    phrase = " ".join(normalized_tokens)
    for ent in doc.ents:
        if ent.text.lower() == phrase:
            return doc

    # token group not found, proceed with creating a new merged entity
    for i in range(len(doc) - len(normalized_tokens) + 1):

        match = True

        for j in range(len(normalized_tokens)):
            if doc[i + j].text.lower() != normalized_tokens[j]:
                match = False
                break

        if match:
            span = doc[i:i+len(normalized_tokens)]

            with doc.retokenize() as retokenizer:
                retokenizer.merge(
                    span,
                    attrs={
                        "ENT_TYPE": entity_type,
                        "ENT_IOB": "B"
                    }
                )

            return doc

    return doc


In [8]:
doc = merge_tokens_to_ner(doc, ["DON", "PEDRO"])
doc = merge_tokens_to_ner(doc, ["DON", "JOHN"])

# Verify the changes
for ent in doc.ents:
    print(f"Entity: {ent.text}, Lemma: {ent.lemma_}, Label: {ent.label_}")


Entity: DON, Lemma: DON, Label: ORG
Entity: DON JOHN, Lemma: DON JOHN, Label: PERSON
Entity: CLAUDIO, Lemma: CLAUDIO, Label: ORG
Entity: BENEDICK, Lemma: BENEDICK, Label: ORG
Entity: Padua, Lemma: Padua, Label: GPE
Entity: LEONATO, Lemma: LEONATO, Label: ORG
Entity: Messina, Lemma: Messina, Label: GPE
Entity: Servant, Lemma: Servant, Label: PERSON
Entity: Don Pedro, Lemma: Don Pedro, Label: PERSON
Entity: BORACHIO, Lemma: BORACHIO, Label: PERSON
Entity: Don John, Lemma: Don John, Label: PERSON
Entity: Don John, Lemma: Don John, Label: PERSON
Entity: Constable, Lemma: Constable, Label: ORG
Entity: VERGES, Lemma: VERGES, Label: ORG
Entity: Headborough, Lemma: Headborough, Label: GPE
Entity: Sexton, Lemma: Sexton, Label: GPE
Entity: Daughter to Leonato, Lemma: daughter to Leonato, Label: PERSON
Entity: Niece, Lemma: Niece, Label: PERSON
Entity: Leonato, Lemma: Leonato, Label: PERSON
Entity: Watch, Attendants, Lemma: Watch, Attendants, Label: ORG


In [9]:
# Here we go through the sentences and continuously fine-tune the model
def display_entities(doc):
    doc_sentences = list(doc.sents)
    print(doc_sentences)
    displacy.render(doc_sentences, style='ent', jupyter=True)

display_entities(doc)

[DON PEDRO is Prince of Arragon., DON JOHN is his bastard Brother., CLAUDIO is a young Lord of Florence., BENEDICK is a young Lord of Padua., LEONATO is Governor of Messina., ANTONIO is his Brother., BALTHASAR is Servant to Don Pedro., BORACHIO is follower of Don John., CONRADE is follower of Don John., DOGBERRY is a Constable., VERGES is a Headborough., FRIAR FRANCIS., is a character., A Sexton., is a character., A Boy. is a character., HERO is Daughter to Leonato., BEATRICE is Niece to Leonato., MARGARET is Waiting gentlewoman attending on Hero., URSULA is Waiting gentlewoman attending on Hero., Messengers is Watch, Attendants, etc.]


C:\Users\drago\AppData\Local\Programs\Python\Python39\lib\site-packages\spacy\displacy\__init__.py:213: UserWarning: [W006] No entities to visualize found in Doc object. If this is surprising to you, make sure the Doc was processed using a model that supports named entity recognition, and check the `doc.ents` property manually if necessary.
  warnings.warn(Warnings.W006)


### Fine-tuning the character information

Based on the output from the above cell, we can now proceed with adding the entities that the default English language model missed. Turns out, this task is two-fold:
- Adding new tokens to existing NER tags
- Adding new NER tags to the language model

The first one is straightforward - we identify words that look like real-world entities, and add them into the language model. We should take care that we are not adding an entity twice, so the top portion of the code in the `merge_tokens_to_ner()` functions first checks if we have found the tokens in the language. New tokens are added only if they do not already exist. This allows for an iterative, robust, fine-tuning strategy that is less error-prone.

In [10]:
# Fine-tuning using existing tag information

# 0-10
doc = merge_tokens_to_ner(doc, ["Arragon"], "GPE")
doc = merge_tokens_to_ner(doc, ["Florence"], "GPE")
doc = merge_tokens_to_ner(doc, ["CLAUDIO"], "PERSON")
doc = merge_tokens_to_ner(doc, ["BENEDICK"], "PERSON")
doc = merge_tokens_to_ner(doc, ["LEONATO"], "PERSON")
doc = merge_tokens_to_ner(doc, ["ANTONIO"], "PERSON")
doc = merge_tokens_to_ner(doc, ["BALTHASAR"], "PERSON")
doc = merge_tokens_to_ner(doc, ["CONRADE"], "PERSON")
doc = merge_tokens_to_ner(doc, ["DOGBERRY"], "PERSON")
doc = merge_tokens_to_ner(doc, ["FRIAR", "FRANCIS"], "PERSON")
doc = merge_tokens_to_ner(doc, ["Boy"], "PERSON")
doc = merge_tokens_to_ner(doc, ["HERO"], "PERSON")
doc = merge_tokens_to_ner(doc, ["BEATRICE"], "PERSON")
doc = merge_tokens_to_ner(doc, ["MARGARET"], "PERSON")
doc = merge_tokens_to_ner(doc, ["URSULA"], "PERSON")
# 11-20 (can be batched for readability in a production environment)


In [11]:
display_entities(doc)

[DON PEDRO is Prince of Arragon., DON JOHN is his bastard Brother., CLAUDIO is a young Lord of Florence., BENEDICK is a young Lord of Padua., LEONATO is Governor of Messina., ANTONIO is his Brother., BALTHASAR is Servant to Don Pedro., BORACHIO is follower of Don John., CONRADE is follower of Don John., DOGBERRY is a Constable., VERGES is a Headborough., FRIAR FRANCIS., is a character., A Sexton., is a character., A Boy. is a character., HERO is Daughter to Leonato., BEATRICE is Niece to Leonato., MARGARET is Waiting gentlewoman attending on Hero., URSULA is Waiting gentlewoman attending on Hero., Messengers is Watch, Attendants, etc.]


Now that we have assigned the existing NER labels to the entities in our input, we turn our attention to words that are either misclassified, or do not have a label category that is most appropriate to solve the Q&A type interaction we envision.

E.g. words such as _'Prince'_, _'Governor'_, _'Lord'_ etc. point to titles held by the nobility that are also the labels of their responsibilities or "OCCUPATION." Similar roles can be found in the tokens _"Waiting gentlewoman"_, _"follower"_, _"Friar"_, and _"Headborough"_.

There is, unfortunately, no label within the default scheme that accurately captures the nuance of titles and occupations. We can however, create one ourselves! Let us define it as follows:
 - "OCC" : titles, occupations, or other work-related roles held by individuals

Now we add this label to appropriate tokens.

In [12]:
# Add code to insert new NER label "OCC" for the appropriate entities
doc = merge_tokens_to_ner(doc, ["Prince"], "OCC")
doc = merge_tokens_to_ner(doc, ["Governor"], "OCC")
doc = merge_tokens_to_ner(doc, ["Lord"], "OCC")
doc = merge_tokens_to_ner(doc, ["Waiting", "gentlewoman"], "OCC")
doc = merge_tokens_to_ner(doc, ["follower"], "OCC")
doc = merge_tokens_to_ner(doc, ["Friar"], "OCC")
doc = merge_tokens_to_ner(doc, ["Headborough"], "OCC")

In [13]:
display_entities(doc)

[DON PEDRO is Prince of Arragon., DON JOHN is his bastard Brother., CLAUDIO is a young Lord of Florence., BENEDICK is a young Lord of Padua., LEONATO is Governor of Messina., ANTONIO is his Brother., BALTHASAR is Servant to Don Pedro., BORACHIO is follower of Don John., CONRADE is follower of Don John., DOGBERRY is a Constable., VERGES is a Headborough., FRIAR FRANCIS., is a character., A Sexton., is a character., A Boy. is a character., HERO is Daughter to Leonato., BEATRICE is Niece to Leonato., MARGARET is Waiting gentlewoman attending on Hero., URSULA is Waiting gentlewoman attending on Hero., Messengers is Watch, Attendants, etc.]


Similarly, we can create another label for "RELationship" - pointing to relatives, family or other ties that exist between people.

In [14]:
# Add code to insert new NER label "REL" for the appropriate entities
doc = merge_tokens_to_ner(doc, ["Brother"], "REL")
doc = merge_tokens_to_ner(doc, ["Niece"], "REL")
doc = merge_tokens_to_ner(doc, ["Daughter"], "REL")
display_entities(doc)

[DON PEDRO is Prince of Arragon., DON JOHN is his bastard Brother., CLAUDIO is a young Lord of Florence., BENEDICK is a young Lord of Padua., LEONATO is Governor of Messina., ANTONIO is his Brother., BALTHASAR is Servant to Don Pedro., BORACHIO is follower of Don John., CONRADE is follower of Don John., DOGBERRY is a Constable., VERGES is a Headborough., FRIAR FRANCIS., is a character., A Sexton., is a character., A Boy. is a character., HERO is Daughter to Leonato., BEATRICE is Niece to Leonato., MARGARET is Waiting gentlewoman attending on Hero., URSULA is Waiting gentlewoman attending on Hero., Messengers is Watch, Attendants, etc.]


That's it! We have successfully converted our input data into a form where we can now derive meaningful relationships between named-entities within. These relationships are either pre-baked, or some that we added in the process of fine-tuning the language model.

Before we proceed to the next portion, we should serialize the document object so that we can cache the work we have done so far.

In [15]:
import gc, json

with open("doc.json", "w") as f:
    json.dump(doc.to_json(), f, indent=4)

gc.collect()

6600